# 04. Tools & Structured Outputs Fundamentals

This notebook teaches the core mechanism of agentic behavior: Tool Calling.

We will explore how to enforce strict JSON schemas, map them to Python functions, and handle execution safely.

## 1. Defining Tools with Pydantic

Instead of writing raw JSON Schemas, modern agent engineering relies on Pydantic to define typed parameters.

In [ ]:
from pydantic import BaseModel, Field

class RefundOrder(BaseModel):
    """Issues a refund for a specific customer order."""
    order_id: str = Field(..., description='The unique identifier for the order')
    amount: float = Field(..., description='The amount to refund in USD')
    reason: str = Field(..., description='The reason for the refund')

print('Pydantic schema generated successfully!')
print(RefundOrder.model_json_schema())

## 2. OpenAI Function Calling

We pass the Pydantic schema to the `tools` parameter of the OpenAI API. The model returns a `tool_calls` array instead of text.

In [ ]:
# Mocked OpenAI API Response
mock_response = {
    'choices': [{
        'message': {
            'role': 'assistant',
            'tool_calls': [{
                'id': 'call_123',
                'type': 'function',
                'function': {
                    'name': 'RefundOrder',
                    'arguments': '{"order_id":"ord_999","amount":50.0,"reason":"defective item"}'
                }
            }]
        }
    }]
}

print('Model decided to call a tool!')
tool_call = mock_response['choices'][0]['message']['tool_calls'][0]
print(f"Function: {tool_call['function']['name']}")
print(f"Arguments: {tool_call['function']['arguments']}")

## 3. Execution, Validation, and Retries

Once the model requests a tool, our application must parse the arguments, validate them against the schema, and execute the actual Python code.

In [ ]:
import json

raw_args = tool_call['function']['arguments']
try:
    # 1. Parse JSON
    parsed_args = json.loads(raw_args)
    # 2. Validate with Pydantic
    validated_request = RefundOrder(**parsed_args)
    print(f'Validation Passed! Issuing refund of ${validated_request.amount} for order {validated_request.order_id}.')
    # 3. Append Tool Result message back to conversation history here...
except Exception as e:
    print(f'Validation Failed! Sending error back to model to retry: {e}')

## 4. PydanticAI (Framework Abstraction)

Writing the raw execution loop is tedious. Frameworks like **PydanticAI** handle the parsing, validation, and retries automatically.

In [ ]:
# Simulated PydanticAI syntax
print('from pydantic_ai import Agent')
print('agent = Agent("openai:gpt-4o")')
print('@agent.tool')
print('def issue_refund(order_id: str, amount: float) -> str:')
print('    return "Refund successful"')
print('result = agent.run_sync("Refund order 123 for $50")')
print('print(result.data)')
print('\nPydanticAI manages the tool loop under the hood.')